In [ ]:
import ROOT
from XRootD.client import FileSystem
import json
import os
from pprint import pprint
from tqdm import tqdm

# Define the sample nick for which the filelist information needs to be fixed
# Other samples modified with this notebook:
# - DYto2Mu-2Jets_Bin-2J-MLL-50_TuneCP5_13p6TeV_amcatnloFXFX-pythia8_RunIII2024Summer24NanoAODv15-150X
# - WW_TuneCP5_13p6TeV_pythia8_RunIII2024Summer24NanoAODv15-150X
# - GluGluH-Hto2TauUncorrelatedDecay_Par-M-125_TuneCP5_13p6TeV_powheg-pythia8_RunIII2024Summer24NanoAODv15-150X
SAMPLE_NICK = "WW_TuneCP5_13p6TeV_pythia8_RunIII2024Summer24NanoAODv15-150X"

In [ ]:
%%capture cap_sample_database

# Get sample metadata from the sample database
jq_string = f"' .[] | select(.nick | startswith(\"{SAMPLE_NICK}\")) '"
!cat ../nanoAOD_v15/datasets.json | jq -r -M {jq_string}

In [ ]:
# Load the sample information into a dictionary
sample_info = json.loads(cap_sample_database.stdout)
pprint(sample_info)

In [ ]:
%%capture cap_sample_database_filelist

# Get sample file metadata from the sample database
filepath = os.path.join("..", "nanoAOD_v15", sample_info["era"], sample_info["sample_type"], f"{sample_info['nick']}.json")
!cat {filepath} | jq -r -M ' . '

In [ ]:
# Load the sample filelist information into a dictionary
sample_info_filelist = json.loads(cap_sample_database_filelist.stdout)
pprint(sample_info_filelist)

In [ ]:
%%capture cap_dasgoclient

# Query files of the dataset and extract file address and number of events
das_string = f"\'file dataset={sample_info['dbs']}\'"
!dasgoclient --query={das_string} -json

In [ ]:
# Load the captured output of the dasgoclient command and compile the filelist
files = [
    {
        "name": entry["file"][0]["name"],
        "nevents": entry["file"][0]["nevents"],
        "status": None,
        "stat_info": None
    }
    for entry in json.loads(cap_dasgoclient.stdout)
]

In [ ]:
# Create the remote file system client
# Other choices:
# - GridKA: root://cmsdcache-kit-disk.gridka.de
# - Gobal: root://xrd-global.cern.ch
remote_fs = FileSystem("root://xrootd-cms.infn.it")
infn_url = "root://xrootd-cms.infn.it//"

for file in tqdm(files, desc="Checking files", unit="files", total=len(files)):
    # Check if the ROOT file is on GridKA
    status, stat_info = remote_fs.stat(file["name"])
    status_code = status.code

    # Explicitly open the file to check if it can be accessed
    if status_code == 0:
        f = None
        try:
            f = ROOT.TFile.Open(infn_url + file["name"], "READ")
            if f is None:
                status_code = -1
            elif f.IsZombie():
                status_code = -1
        except Exception:
            status_code = -1
        finally:
            if isinstance(f, ROOT.TFile) and f.IsOpen():
                f.Close()

    # Add information to file dict
    file["status_code"] = status_code
    file["stat_info"] = stat_info

# Build list of available files
available_files = list(sorted((file for file in files if file["status_code"] == 0), key=lambda x: x["name"]))

# Print summary of available files and events
nevents = sum(file["nevents"] for file in files)
nfiles = len(files)
available_nevents = sum(file["nevents"] for file in available_files)
available_nfiles = len(available_files)
print(f"Summary for {SAMPLE_NICK}:")
print(f"    Available files:  {available_nfiles}/{nfiles}")
print(f"    Available events: {available_nevents}/{nevents}")

In [ ]:
# Update information in the sample database
with open("../nanoAOD_v15/datasets.json", "r") as f:
    datasets = json.load(f)
datasets[SAMPLE_NICK]["nevents"] = available_nevents
datasets[SAMPLE_NICK]["nfiles"] = available_nfiles
with open("../nanoAOD_v15/datasets.json", "w") as f:
    json.dump(datasets, f, indent=4, sort_keys=True)

In [ ]:
# Update information in the filelist file 
sample_file = f"../nanoAOD_v15/{sample_info['era']}/{sample_info['sample_type']}/{SAMPLE_NICK}.json"
with open(sample_file, "r") as f:
    dataset = json.load(f)
dataset["filelist"] = [infn_url + file["name"] for file in available_files]
dataset["nevents"] = available_nevents
dataset["nfiles"] = available_nfiles
with open(sample_file, "w") as f:
    json.dump(dataset, f, indent=4, sort_keys=True)